# 03 · MGAM partition 탐색과 누적 ablation

연속 fragment의 $2^{n-1}$ partition을 열거해 reference granularity에 가장 잘 맞는 구성을 고른다. 공식 MGAM evaluator의 교육용 축약판이다.

**학습 목표**: bit mask로 모든 연속 partition을 열거하고 granularity가 다른 block의 최적 대응을 찾는다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 외부 패키지는 없으며 Python 표준 기능만 사용한다.

In [ ]:
# 경계마다 자를지 여부를 bit로 표현하면 정확히 2^(n-1) partition을 빠짐없이 만든다.
def levenshtein(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        new = [i]
        for j, cb in enumerate(b, 1):
            new.append(min(new[-1] + 1, row[j] + 1, row[j - 1] + (ca != cb)))
        row = new
    return row[-1]

def contiguous_partitions(fragments):
    for mask in range(1 << (len(fragments) - 1)):
        groups, current = [], [fragments[0]]
        for boundary, fragment in enumerate(fragments[1:]):
            if mask & (1 << boundary):
                groups.append(' '.join(current))
                current = [fragment]
            else:
                current.append(fragment)
        groups.append(' '.join(current))
        yield groups

def partition_cost(reference, groups):
    if len(reference) != len(groups):
        return 1.0 + abs(len(reference) - len(groups))
    return sum(levenshtein(a, b) / max(1, len(a), len(b)) for a, b in zip(reference, groups)) / len(reference)


In [ ]:
reference = ['total revenue', '42 million']
fragments = ['total', 'revenue', '42', 'million']
candidates = [(partition_cost(reference, groups), groups) for groups in contiguous_partitions(fragments)]
best_cost, best_groups = min(candidates, key=lambda item: item[0])
print('partition count:', len(candidates))
print('best:', best_cost, best_groups)
assert len(candidates) == 2 ** (len(fragments) - 1)
assert best_cost == 0.0

scores = [('baseline', 92.98), ('+ Stage 1', 94.29), ('+ Stage 2', 95.25), ('+ Stage 3', 95.69)]
for (name, score), (_, previous) in zip(scores[1:], scores[:-1]):
    print(name, 'increment=', round(score - previous, 2))


partition 수는 fragment 수에 지수적으로 증가한다. production evaluator는 탐색 한도와 fallback을 기록해야 하며, 누적 ablation의 증분을 각 stage의 단독 효과로 해석해서는 안 된다.